# Лабораторная 5
Финальные метрики и ablation по сохранённым данным. Команда живого прогона из корня: `python -m ml.final_eval --split all --output docs/experiments/lab5-run`.

In [1]:
import json
from pathlib import Path
run = Path("lab5-run")
if not run.exists():
    run = Path("docs/experiments/lab5-run")
metrics = {}
selection = json.loads((run/"selection.json").read_text(encoding="utf-8"))
print("Рабочий вариант:", selection["selected_variant"], selection["service_version"])
for name in ("delivery_before", "final", "no_prompt", "no_calendar", "no_guards"):
    metrics[name] = json.loads((run/name/"metrics.json").read_text(encoding="utf-8"))
    print(name, {k: f"{metrics[name][k]['k']}/{metrics[name][k]['n']}"
                 for k in ("M2", "M3", "M6", "E3", "S1")})

Рабочий вариант: no_prompt lab5-guarded
delivery_before {'M2': '43/53', 'M3': '9/53', 'M6': '13/59', 'E3': '52/53', 'S1': '10/10'}
final {'M2': '41/53', 'M3': '10/53', 'M6': '14/59', 'E3': '52/53', 'S1': '9/10'}
no_prompt {'M2': '37/53', 'M3': '13/53', 'M6': '12/59', 'E3': '52/53', 'S1': '10/10'}
no_calendar {'M2': '41/53', 'M3': '10/53', 'M6': '14/59', 'E3': '52/53', 'S1': '9/10'}
no_guards {'M2': '45/53', 'M3': '7/53', 'M6': '15/59', 'E3': '52/53', 'S1': '9/10'}


In [2]:
for name in ("no_prompt", "no_calendar", "no_guards"):
    print(name, "M2 contribution:", metrics[name]["M2"]["k"] - metrics["final"]["M2"]["k"])
for name in metrics:
    rows = [json.loads(line) for line in (run/name/"requests.jsonl").read_text(encoding="utf-8").splitlines()]
    print(name, "observations:", len(rows), "known tokens:",
          sum(r.get("usage", {}).get("input_tokens",0)+r.get("usage", {}).get("output_tokens",0) for r in rows))
    assert len(rows) == 55

no_prompt M2 contribution: -4
no_calendar M2 contribution: 0
no_guards M2 contribution: 4
delivery_before observations: 55 known tokens: 26470
final observations: 55 known tokens: 32764
no_prompt observations: 55 known tokens: 26545
no_calendar observations: 55 known tokens: 32692
no_guards observations: 55 known tokens: 32764


Отрицательный вклад — ухудшение. Выборка и модель отличаются от лаб 3–4. Полный денежный TCO и реальные публикации не измерены.